Import libraries

In [ ]:
!pip install opencv-python scikit-image scikit-learn numpy

import os
import cv2
import numpy as np
from google.colab import drive
from skimage import feature
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import SelectKBest, mutual_info_classif

Load Dataset



In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
import zipfile

zip_path = "/content/drive/MyDrive/archive .zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content/")

print("Unzipped successfully!")

Unzipped successfully!


In [10]:
import os
dataset_path = "/content/plantvillage dataset/color"
print("Number of classes:", len(os.listdir(dataset_path)))

Number of classes: 38


In [15]:
images = []
labels = []

for class_name in os.listdir(dataset_path):
    class_path = os.path.join(dataset_path, class_name)

    if os.path.isdir(class_path):
        for file in os.listdir(class_path):
            img_path = os.path.join(class_path, file)
            img = cv2.imread(img_path)

            if img is not None:
                # Required preprocessing for feature extraction
                img = cv2.resize(img, (128, 128))
                img = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

                images.append(img)
                labels.append(class_name)

images = np.array(images)
labels = np.array(labels)

print("Total images:", len(images))
print("Image shape:", images.shape)

Total images: 54305
Image shape: (54305, 128, 128, 3)


Feature extraction

In [18]:
def extract_features(image):
    # 1. HSV Color Histogram
    hist_h = cv2.calcHist([image], [0], None, [256], [0, 256])
    hist_s = cv2.calcHist([image], [1], None, [256], [0, 256])
    hist_v = cv2.calcHist([image], [2], None, [256], [0, 256])

    hist_h = cv2.normalize(hist_h, None, alpha=1.0,
                           norm_type=cv2.NORM_L1).flatten()
    hist_s = cv2.normalize(hist_s, None, alpha=1.0,
                           norm_type=cv2.NORM_L1).flatten()
    hist_v = cv2.normalize(hist_v, None, alpha=1.0,
                           norm_type=cv2.NORM_L1).flatten()

    hsv_hist = np.concatenate([hist_h, hist_s, hist_v])

    # 2. Local Binary Pattern (LBP)
    gray_image = image[:, :, 2]
    radius = 3
    n_points = 8 * radius

    lbp = feature.local_binary_pattern(
        gray_image, n_points, radius, method="uniform"
    )

    hist_lbp, _ = np.histogram(
        lbp.ravel(),
        bins=np.arange(0, n_points + 3),
        range=(0, n_points + 2),
    )

    hist_lbp = cv2.normalize(
        hist_lbp.astype(np.float32),
        None,
        alpha=1.0,
        norm_type=cv2.NORM_L1
    ).flatten()

    # 3. Hu Moments
    _, binary = cv2.threshold(
        gray_image, 0, 255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )
    moments = cv2.moments(binary)
    hu_moments = cv2.HuMoments(moments).flatten()

    # Combine all features
    return np.concatenate([hsv_hist, hist_lbp, hu_moments])

In [20]:
from skimage import feature
X_features = []
for img in images:
    X_features.append(extract_features(img))

X_features = np.array(X_features)

print("Shape of extracted features:", X_features.shape)

Shape of extracted features: (54305, 801)


Feature Selection (Mutual Information)

In [22]:
from sklearn.feature_selection import SelectKBest, mutual_info_classif

# Select top K features based on Mutual Information
k_features = 100 # You can adjust this number
selector = SelectKBest(mutual_info_classif, k=k_features)
X_selected = selector.fit_transform(X_features, labels)

print(f"Original feature shape: {X_features.shape}")
print(f"Selected feature shape: {X_selected.shape}")

Original feature shape: (54305, 801)
Selected feature shape: (54305, 100)


In [24]:
np.savez("plant_disease_features.npz", X_features_selected=X_selected, y_labels=labels)

In [25]:
from google.colab import files

files.download("plant_disease_features.npz")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>